# ERA5 wind data for any site

Pick a site on a map, type its position into one cell, and this notebook downloads one year of hourly ERA5 weather data for that spot. It then writes a CSV file with the same layout as `wind_data_template 7.csv`: one row per hour with wind speed, wind direction, air temperature and air pressure, all at 100 m above the ground.

## Before you start

Install the packages once. Open a terminal and run:

```
pip install cdsapi cfgrib eccodes xarray pandas numpy folium jupyterlab
```

The data comes from the Copernicus Climate Data Store (CDS), which needs a free account:

1. Register at https://cds.climate.copernicus.eu and log in.
2. Open the dataset page for "ERA5 hourly data on single levels", go to the Download tab and accept the licence at the bottom. You only need to do this once. Without it every download fails.
3. Open your profile page (click your name at the top right) and copy your API key.

## 1. Settings

Paste your API key between the quotes. Leave the rest as it is unless your teacher tells you otherwise.

In [ ]:
API_KEY = "paste-your-key-here"

YEAR = 2025
SITE_ID = 1         # label written in the first line of the CSV
TIMEZONE = 1        # time zone of the site in hours from UTC (Denmark is +1)
DATA_DIR = "era5_data"

## 2. Download function

This cell only defines the function. Nothing is downloaded yet.

The CDS does not allow a whole year in one request, so the function asks for one month at a time and saves 12 small GRIB files in the `era5_data` folder. Each request covers a 1° by 1° box around the site. Requests wait in a queue at the CDS, so each month can take a few minutes.

In [ ]:
import time
from pathlib import Path

import cdsapi


def download_era5(lat, lon, year):
    """Download one year of hourly ERA5 data around (lat, lon), one GRIB file per month."""
    if API_KEY == "paste-your-key-here":
        raise ValueError("Paste your API key in the settings cell (step 1) and run that cell first.")
    client = cdsapi.Client(url="https://cds.climate.copernicus.eu/api", key=API_KEY, quiet=True, progress=False)
    Path(DATA_DIR).mkdir(exist_ok=True)
    files = []
    start = time.time()

    for month in range(1, 13):
        file = Path(DATA_DIR) / f"era5_{lat}_{lon}_{year}_{month:02d}.grib"
        files.append(file)
        if file.exists():
            print(f"Month {month:2d}/12: already downloaded")
            continue

        print(f"Month {month:2d}/12: waiting for the CDS ...", end=" ", flush=True)
        request = {
            "product_type": ["reanalysis"],
            "variable": [
                "100m_u_component_of_wind",
                "100m_v_component_of_wind",
                "2m_temperature",
                "surface_pressure",
            ],
            "year": [str(year)],
            "month": [f"{month:02d}"],
            "day": [f"{d:02d}" for d in range(1, 32)],
            "time": [f"{h:02d}:00" for h in range(24)],
            "area": [lat + 0.5, lon - 0.5, lat - 0.5, lon + 0.5],  # north, west, south, east
            "data_format": "grib",
            "download_format": "unarchived",
        }
        part = file.with_suffix(".part")
        client.retrieve("reanalysis-era5-single-levels", request).download(str(part))
        part.rename(file)
        minutes = (time.time() - start) / 60
        print(f"done. {month / 12:.0%} finished after {minutes:.0f} min")

    return files

## 3. Conversion function

This cell also only defines a function. It opens the 12 monthly files, picks the ERA5 grid point of the site and computes the four columns.

ERA5 does not store these four values directly, so we compute them from what it has.

ERA5 stores the wind as two components: `u100` blows towards the east and `v100` towards the north. Wind speed is $\sqrt{u^2 + v^2}$. Wind direction is the direction the wind comes from, measured clockwise from north, so 0° is a north wind and 90° is an east wind.

The air temperature `t2m` is measured at 2 m. To estimate it at 100 m we subtract 0.0065 °C for every metre of height, the average rate in the standard atmosphere. That makes the air at 100 m about 0.6 °C colder.

The pressure `sp` is measured at the ground. The barometric formula $p_{100} = p_{surface} \cdot e^{-g h / (R T)}$ gives the pressure 100 m higher, which comes out about 1.2 % lower.

The CSV starts with a line of site information, then the column names, like the template. ERA5 times are in UTC, so the data time zone is 0 and the first row is 1 January 00:00 UTC.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr


def era5_to_csv(files, lat, lon, csv_file):
    """Turn the monthly ERA5 GRIB files into a wind resource CSV for the grid point nearest (lat, lon)."""
    months = [xr.open_dataset(f, engine="cfgrib", backend_kwargs={"indexpath": ""}) for f in files]
    ds = xr.concat(months, dim="time")
    ds = ds.assign_coords(longitude=(ds.longitude + 180) % 360 - 180).sortby("longitude")  # same -180..180 as the map
    point = ds.sel(latitude=lat, longitude=lon, method="nearest")
    grid_lat, grid_lon = float(point.latitude), float(point.longitude)
    if abs(grid_lat - lat) > 0.2 or abs(grid_lon - lon) > 0.2:
        raise ValueError(f"Nearest grid point {grid_lat}, {grid_lon} is far from the site. Check the files in {DATA_DIR}.")

    u = point["u100"].values    # m/s
    v = point["v100"].values    # m/s
    t2m = point["t2m"].values   # kelvin
    sp = point["sp"].values     # pascal

    wind_speed = np.sqrt(u**2 + v**2)
    wind_direction = np.degrees(np.arctan2(-u, -v)) % 360
    temp_100m = t2m - 0.0065 * (100 - 2)
    pressure_100m = sp * np.exp(-9.81 * 100 / (287.05 * temp_100m))

    df = pd.DataFrame(
        {
            "wind speed at 100m (m/s)": wind_speed.round(2),
            "wind direction at 100m (deg)": wind_direction.round(2),
            "air temperature at 100m (C)": (temp_100m - 273.15).round(2),
            "air pressure at 100m (Pa)": pressure_100m.round().astype(int),
        },
        index=point["time"].values,
    )

    header = (
        f"SiteID,{SITE_ID},Site Timezone,{TIMEZONE},Data Timezone,0,"
        f"Longitude,{grid_lon},Latitude,{grid_lat}\n"
    )
    with open(csv_file, "w", newline="") as f:
        f.write(header)
        df.to_csv(f, index=False)

    print(f"Saved {csv_file}: {len(df)} hours (a normal year has 8760) for grid point {grid_lat} N, {grid_lon} E")
    return df

## 4. Find your site on the map

Run the cell below and a map appears. Zoom in and click where you want your site. A small box pops up with the latitude and longitude.

In [ ]:
import folium

site_map = folium.Map(location=[56.0, 10.5], zoom_start=6)
site_map.add_child(folium.LatLngPopup())
site_map

## 5. Download and make the CSV

Type the two numbers from the map popup below and run the cell. Progress messages appear under the cell, one line per month.

ERA5 only has data every 0.25° (about 28 km north to south and 16 km east to west in Denmark), so the cell moves your site to the nearest ERA5 grid point and prints which one it uses. The map and ERA5 both give positions in degrees of latitude and longitude (WGS84), so you can copy the numbers straight across.

Keep the notebook open until it says the CSV is saved. If it stops halfway, for example because your laptop went to sleep, run the cell again. Months that are already downloaded are skipped.

If it fails straight away, check that your API key is pasted correctly in step 1 and that you accepted the licence (see "Before you start").

In [ ]:
LAT = 55.52   # latitude from the map
LON = 7.91    # longitude from the map

lat = round(LAT * 4) / 4                          # snap to the 0.25 degree ERA5 grid
lon = round(((LON + 180) % 360 - 180) * 4) / 4    # the map repeats sideways, this keeps lon between -180 and 180
print(f"Using ERA5 grid point {lat} N, {lon} E for {YEAR}")

files = download_era5(lat, lon, YEAR)
df = era5_to_csv(files, lat, lon, f"wind_data_{lat}_{lon}_{YEAR}.csv")
df.describe()

## Changing the notebook

For another site, change `LAT` and `LON` in step 5 and run that cell again. For another year, change `YEAR` in step 1. Each site and year gets its own file names, so nothing is overwritten.

ERA5 single levels also has wind at 10 m (`10m_u_component_of_wind` and `10m_v_component_of_wind`, called `u10` and `v10` in the file). To use it, swap the variable names in steps 2 and 3, and change the height 100 in the temperature and pressure formulas to 10.